In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Reused from Lab 1
def pairwise_distances(X, Y):
    """Computes the Euclidean distance matrix between X (N, D) and Y (M, D)."""
    return np.sqrt(
        np.sum(X**2, axis=1, keepdims=True)
        + np.sum(Y**2, axis=1)
        - 2 * np.dot(X, Y.T)
    )

# Reused from Lab 5
def standardise(X_train, X_test):
    """Standardises X_train and X_test using mean and std computed from X_train only."""
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)
    Z_train = (X_train - mu) / sigma
    Z_test = (X_test - mu) / sigma
    return Z_train, Z_test

In [ ]:
df = pd.read_csv("ic272_lab6_vibration.csv")
features = ["vib_x", "vib_y", "machine_hours"]
X = df[features].to_numpy(float)
y = df["faulty"].to_numpy()

idx = np.random.default_rng(0).permutation(len(y))
te, tr = idx[:100], idx[100:]
X_train, y_train = X[tr], y[tr]
X_test, y_test = X[te], y[te]

print("train:", X_train.shape, "test:", X_test.shape)  # Expected: (300, 3) (100, 3)

In [ ]:
# Check training set standard deviations
for name, s in zip(features, X_train.std(axis=0)):
    print(f"{name:15s}{s:12.3f}")

# (a) Standardise training and test sets
Z_train, Z_test = standardise(X_train, X_test)  # computed from training set only

# (b) Test accuracy at k=7 on raw vs standardised features
raw_preds_7 = knn_predict(X_train, y_train, X_test, k=7)
raw_acc = np.mean(raw_preds_7 == y_test)

std_preds_7 = knn_predict(Z_train, y_train, Z_test, k=7)
std_acc = np.mean(std_preds_7 == y_test)

print(f"\nTest Accuracy at k=7 (Raw features): {raw_acc:.4f}")
print(f"Test Accuracy at k=7 (Standardised features): {std_acc:.4f}")

In [ ]:
def k_fold_cv_knn(X, y, k, folds=5, seed=0):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(y))
    fold_sizes = np.full(folds, len(y) // folds)
    fold_sizes[:len(y) % folds] += 1

    accuracies = []
    current = 0
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        val_idx = indices[start:stop]
        train_idx = np.concatenate([indices[:start], indices[stop:]])

        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # Standardise fold data using fold's training statistics
        mu = np.mean(X_tr, axis=0)
        sigma = np.std(X_tr, axis=0)
        X_tr_std = (X_tr - mu) / sigma
        X_val_std = (X_val - mu) / sigma

        preds = knn_predict(X_tr_std, y_tr, X_val_std, k)
        acc = np.mean(preds == y_val)
        accuracies.append(acc)

        current = stop

    return np.mean(accuracies)

# (b) Sweep over k values and print table
k_list = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51, 75]
cv_results = []
for k_val in k_list:
    cv_acc = k_fold_cv_knn(X_train, y_train, k_val, folds=5, seed=0)
    cv_results.append((k_val, cv_acc))

cv_df = pd.DataFrame(cv_results, columns=["k", "CV_Accuracy"])
print(cv_df)

best_row = cv_df.loc[cv_df["CV_Accuracy"].idxmax()]
best_k = int(best_row["k"])
print(f"\nBest k: {best_k} with CV Accuracy: {best_row['CV_Accuracy']:.4f}")

# (c) Report test accuracy at chosen k, k=1, and k=75
print("\nTest Accuracies:")
for k_val in [best_k, 1, 75]:
    preds = knn_predict(Z_train, y_train, Z_test, k=k_val)
    acc = np.mean(preds == y_test)
    print(f"k = {k_val:2d} -> Test Accuracy: {acc:.4f}")

In [ ]:
# --- H1: The curse of dimensionality ---
e_values = [0, 2, 5, 10, 20, 50]
h1_accuracies = []
h1_ratios = []

X_train_2d = X_train[:, :2]
X_test_2d = X_test[:, :2]

rng_noise = np.random.default_rng(0)
for e in e_values:
    if e == 0:
        tr_noise = X_train_2d
        te_noise = X_test_2d
    else:
        noise_tr = rng_noise.normal(0, 1, size=(X_train_2d.shape[0], e))
        noise_te = rng_noise.normal(0, 1, size=(X_test_2d.shape[0], e))
        tr_noise = np.hstack([X_train_2d, noise_tr])
        te_noise = np.hstack([X_test_2d, noise_te])

    mu = np.mean(tr_noise, axis=0)
    sigma = np.std(tr_noise, axis=0)
    tr_std = (tr_noise - mu) / sigma
    te_std = (te_noise - mu) / sigma

    preds = knn_predict(tr_std, y_train, te_std, k=7)
    acc = np.mean(preds == y_test)
    h1_accuracies.append(acc)

    dists = pairwise_distances(te_std, tr_std)
    min_d = np.min(dists)
    max_d = np.max(dists)
    h1_ratios.append(min_d / max_d)

print("H1 Completed. Accuracies:", h1_accuracies)
print("H1 Distance Ratios:", h1_ratios)

# --- H2: Distance-weighted voting ---
def knn_predict_weighted(X_train, y_train, X_query, k, epsilon=1e-5):
    dists = pairwise_distances(X_query, X_train)
    nearest_indices = np.argsort(dists, axis=1)[:, :k]
    nearest_dists = np.take_along_axis(dists, nearest_indices, axis=1)
    neighbor_labels = y_train[nearest_indices]
    weights = 1.0 / (nearest_dists + epsilon)
    weighted_votes = np.sum(neighbor_labels * weights, axis=1) / np.sum(weights, axis=1)
    return (weighted_votes > 0.5).astype(int)

# --- H3: Beyond accuracy ---
def confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

preds_chosen = knn_predict(Z_train, y_train, Z_test, k=best_k)
cm = confusion_matrix(y_test, preds_chosen)
tn, fp, fn, tp = cm.ravel()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nH3 Metrics for Faulty Class (k={best_k}):")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")

# --- H4: The decision boundary ---
x_min, x_max = Z_train[:, 0].min() - 1, Z_train[:, 0].max() + 1
y_min, y_max = Z_train[:, 1].min() - 1, Z_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = np.c_[xx.ravel(), yy.ravel()]
Z_train_2feat = Z_train[:, :2]

preds_k1 = knn_predict(Z_train_2feat, y_train, grid_points, k=1).reshape(xx.shape)
preds_bk = knn_predict(Z_train_2feat, y_train, grid_points, k=best_k).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].contourf(xx, yy, preds_k1, alpha=0.3, cmap=plt.cm.coolwarm)
axes[0].scatter(Z_train_2feat[:, 0], Z_train_2feat[:, 1], c=y_train, edgecolors='k', cmap=plt.cm.coolwarm)
axes[0].set_title("Decision Boundary at k=1 (Ragged)")

axes[1].contourf(xx, yy, preds_bk, alpha=0.3, cmap=plt.cm.coolwarm)
axes[1].scatter(Z_train_2feat[:, 0], Z_train_2feat[:, 1], c=y_train, edgecolors='k', cmap=plt.cm.coolwarm)
axes[1].set_title(f"Decision Boundary at chosen k={best_k} (Smooth)")
plt.tight_layout()
plt.show()